# column

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook
import gc

# ============================================================
# SHOW COLUMNS FIRST ONLY
# No save
# No chart
# No cleaning
# ============================================================

INPUT_FILE = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "forecast_projection_occupation.xlsx"
)

print("=" * 100)
print("INPUT FILE:")
print(INPUT_FILE)
print("=" * 100)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

# ============================================================
# SHOW SHEET NAMES
# ============================================================

wb = load_workbook(INPUT_FILE, read_only=True, data_only=True)

print("SHEETS FOUND:")
for i, sheet_name in enumerate(wb.sheetnames, start=1):
    print(f"{i}. {sheet_name}")

print("=" * 100)

# ============================================================
# SHOW COLUMNS USING PANDAS NORMAL HEADER
# ============================================================

for sheet_name in wb.sheetnames:
    print("\n" + "=" * 100)
    print("SHEET:")
    print(sheet_name)
    print("=" * 100)

    try:
        df_head = pd.read_excel(INPUT_FILE, sheet_name=sheet_name, nrows=5)

        print("PANDAS DETECTED COLUMNS:")
        print("Columns count:", len(df_head.columns))
        print()

        for i, col in enumerate(df_head.columns, start=1):
            print(f"{i:3}. {col}")

        print("\nFIRST 5 ROWS PREVIEW:")
        print(df_head.to_string(index=False))

    except Exception as e:
        print("Could not read with pandas normal header.")
        print("Error:", e)

    gc.collect()

# ============================================================
# SHOW RAW FIRST 30 ROWS
# This helps if real header is not row 1
# ============================================================

print("\n" + "=" * 100)
print("RAW EXCEL FIRST 30 ROWS")
print("Use this if pandas columns look wrong.")
print("=" * 100)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]

    print("\n" + "=" * 100)
    print("RAW SHEET:")
    print(sheet_name)
    print("=" * 100)

    for row_number, row in enumerate(ws.iter_rows(min_row=1, max_row=30, values_only=True), start=1):
        values = []
        for col_number, value in enumerate(row, start=1):
            if value is not None:
                values.append(f"C{col_number}={value}")

        if values:
            print(f"ROW {row_number}:")
            print(" | ".join(values))
            print("-" * 100)

wb.close()
gc.collect()

print("\n" + "=" * 100)
print("DONE - column scan only. No file saved.")
print("=" * 100)

# image

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import gc
import time

# ============================================================
# IMAGE ONLY
# PROJECTED JOB OPENINGS BY OCCUPATION SECTOR
#
# INPUT:
# /Users/YourUserName123/Downloads/Internship_SCIPE CI-SIP/MainProject/2_Job/forecast_projection_occupation.xlsx
#
# OUTPUT IMAGE ONLY:
# /Users/YourUserName123/Downloads/Overall_projected_job_openings_by_occupation_sector.png
# ============================================================

INPUT_FILE = (
    Path.home()
    / "Downloads"
    / "Internship_SCIPE CI-SIP"
    / "MainProject"
    / "2_Job"
    / "forecast_projection_occupation.xlsx"
)

OUTPUT_IMAGE = (
    Path.home()
    / "Downloads"
    / "Overall_projected_job_openings_by_occupation_sector.png"
)

SHEET_NAME = "Table 1.10"

print("=" * 100)
print("INPUT FILE:")
print(INPUT_FILE)
print()
print("OUTPUT IMAGE:")
print(OUTPUT_IMAGE)
print("=" * 100)

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"File not found: {INPUT_FILE}")

start_time = time.time()

# ============================================================
# READ TABLE 1.10
# IMPORTANT:
# Real column headers are on Excel row 2, so use header=1
# ============================================================

df = pd.read_excel(
    INPUT_FILE,
    sheet_name=SHEET_NAME,
    header=1
)

print("COLUMNS READ:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2}. {col}")
print("=" * 100)

# ============================================================
# RENAME COLUMNS
# ============================================================

df = df.rename(columns={
    "2024 National Employment Matrix title": "occ_title",
    "2024 National Employment Matrix code": "occ_code",
    "Occupation type": "occupation_type",
    "Occupational openings, 2024–34 annual average": "annual_openings_thousands"
})

needed_cols = [
    "occ_title",
    "occ_code",
    "occupation_type",
    "annual_openings_thousands"
]

missing_cols = [col for col in needed_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# ============================================================
# CLEAN
# ============================================================

df = df[needed_cols].copy()

df["occ_title"] = df["occ_title"].astype(str).str.strip()
df["occ_code"] = df["occ_code"].astype(str).str.strip()

df["annual_openings_thousands"] = (
    df["annual_openings_thousands"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .replace(["*", "**", "#", "nan", "NaN", "None", "", "—"], pd.NA)
)

df["annual_openings_thousands"] = pd.to_numeric(
    df["annual_openings_thousands"],
    errors="coerce"
)

df = df.dropna(subset=["occ_title", "occ_code", "annual_openings_thousands"])

# ============================================================
# KEEP ONLY MAJOR OCCUPATION SECTORS
# Example:
# 11-0000 Management occupations
# 13-0000 Business and financial operations occupations
#
# Remove 00-0000 Total all occupations
# ============================================================

sector_df = df[df["occ_code"].str.match(r"^\d{2}-0000$", na=False)].copy()
sector_df = sector_df[sector_df["occ_code"] != "00-0000"].copy()

# Convert from thousands to real number of openings
sector_df["annual_openings"] = sector_df["annual_openings_thousands"] * 1000

sector_df = sector_df.sort_values(
    "annual_openings",
    ascending=False
).reset_index(drop=True)

sector_df.insert(0, "rank", range(1, len(sector_df) + 1))

print("PROJECTED ANNUAL JOB OPENINGS BY OCCUPATION SECTOR")
print(sector_df[["rank", "occ_code", "occ_title", "annual_openings"]].to_string(index=False))
print("=" * 100)

# ============================================================
# MAKE SIDEWAY BAR CHART
# IMAGE ONLY
# ============================================================

plot_df = sector_df.copy()

plot_df["sector_short"] = (
    plot_df["occ_title"]
    .str.replace("Occupations", "Occ.", regex=False)
    .str.replace("occupations", "occ.", regex=False)
    .str.replace("and", "&", regex=False)
)

# Biggest appears at top
plot_df = plot_df.sort_values("annual_openings", ascending=True).reset_index(drop=True)

plt.figure(figsize=(18, 12))

cmap = plt.get_cmap("tab20")
colors = [cmap(i % cmap.N) for i in range(len(plot_df))]

bars = plt.barh(
    plot_df["sector_short"],
    plot_df["annual_openings"],
    color=colors,
    edgecolor="black",
    linewidth=0.4
)

plt.title(
    "Projected Annual Job Openings by Occupation Sector, 2024-2034",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Projected Annual Job Openings")
plt.ylabel("Occupation Sector")
plt.grid(axis="x", alpha=0.3)

plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"{x/1_000_000:.1f}M")
)

max_value = plot_df["annual_openings"].max()
plt.xlim(0, max_value * 1.18)

for bar in bars:
    value = bar.get_width()
    plt.text(
        value,
        bar.get_y() + bar.get_height() / 2,
        f" {value/1_000_000:.1f}M",
        va="center",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(OUTPUT_IMAGE, dpi=300, bbox_inches="tight")
plt.show()

print("=" * 100)
print("DONE")
print("Image saved to:")
print(OUTPUT_IMAGE)
print("No CSV saved. Image only.")
print("Minutes used:", round((time.time() - start_time) / 60, 2))
print("=" * 100)

gc.collect()